# Pratica — Lezione 12. Casi d'uso reali

In classe hai visto il mini-AlphaZero per il Tris: autogioco, ricerca tipo Monte Carlo Tree Search (MCTS, dall'inglese *Monte Carlo Tree Search*) e miglioramento iterativo della politica. In questa pratica vai oltre: prima programmi tu stesso, da zero, il cuore di quella ricerca; poi adatti tutto lo scheletro a un gioco diverso, Forza 4 ridotto.

**Due esercizi:**

1. Completa il passo di selezione ed espansione di MCTS, usando *Upper Confidence bound for Trees* (UCT)
2. Adatta la scacchiera e le regole di vittoria a una versione ridotta di Forza 4, e confronta quante simulazioni servono per smettere di perdere contro un giocatore casuale

**Prima di iniziare, esegui le due celle seguenti per avere pronta la struttura dell'albero di ricerca.** Non serve modificarle.

In [ ]:
import math
import random

import numpy as np

np.random.seed(0)
random.seed(0)


In [ ]:
# --- Tres en Raya: identico al de la clase ---
LINEAS_GANADORAS = [
    (0, 1, 2), (3, 4, 5), (6, 7, 8),
    (0, 3, 6), (1, 4, 7), (2, 5, 8),
    (0, 4, 8), (2, 4, 6),
]


def tablero_nuevo():
    return tuple([0] * 9)  # 0 = vacio, 1 = X, -1 = O


def movimientos_validos(tablero):
    return [i for i, v in enumerate(tablero) if v == 0]


def aplicar_movimiento(tablero, pos, jugador):
    nuevo = list(tablero)
    nuevo[pos] = jugador
    return tuple(nuevo)


def resultado(tablero):
    """Devuelve +1 si gana X, -1 si gana O, 0 si hay empate, None si sigue."""
    for a, b, c in LINEAS_GANADORAS:
        suma = tablero[a] + tablero[b] + tablero[c]
        if suma == 3:
            return 1
        if suma == -3:
            return -1
    if 0 not in tablero:
        return 0
    return None


def heuristica(tablero):
    """Sustituta minima de la red de valor de AlphaZero."""
    puntuacion = 0
    for linea in LINEAS_GANADORAS:
        valores = [tablero[i] for i in linea]
        piezas_x = valores.count(1)
        piezas_o = valores.count(-1)
        if piezas_o == 0:
            puntuacion += piezas_x ** 2
        if piezas_x == 0:
            puntuacion -= piezas_o ** 2
    return math.tanh(puntuacion / 6.0)


tabla_valores = {}


def valor_estimado(tablero):
    if tablero in tabla_valores:
        return tabla_valores[tablero]
    return heuristica(tablero)


# --- El nodo del arbol de busqueda: ya creado, no hace falta tocarlo ---
class Nodo:
    __slots__ = ("tablero", "jugador", "hijos", "N", "W", "expandido")

    def __init__(self, tablero, jugador):
        self.tablero = tablero
        self.jugador = jugador  # jugador (+1 o -1) que debe mover aqui
        self.hijos = {}
        self.N = 0
        self.W = 0.0
        self.expandido = False

    def valor_medio(self):
        return self.W / self.N if self.N > 0 else 0.0


def ucb_score(padre, hijo, c=1.4):
    """Formula UCT: combina Q (que tan bueno parece) con U (bono de
    exploracion que crece si el hijo se ha visitado poco)."""
    q = hijo.valor_medio() * padre.jugador
    u = c * math.sqrt(math.log(padre.N + 1) / (hijo.N + 1))
    return q + u


## Esercizio 1 — Completa la selezione e l'espansione di MCTS

Hai già lo scheletro: il `Nodo`, la formula `ucb_score` (UCT) e le regole del Tris. Ti manca la funzione più importante di tutte, `simular`, che esegue un ciclo completo di selezione, espansione, valutazione e retropropagazione.

Qui sotto trovi `simular` con due spazi contrassegnati con `# TODO`. Nota ciò che è già risolto: se la partita è già terminata (`resultado` non è `None`), il nodo aggiorna `N` e `W` con il risultato reale e non serve toccare nulla lì.

**Il tuo compito è completare i due `# TODO`:**

1. **Espansione** (dentro `if not nodo.expandido:`): crea un `Nodo` figlio per ogni mossa valida a partire da `nodo.tablero`, e salvalo in `nodo.hijos`. Non dimenticare di impostare `nodo.expandido = True` alla fine
2. **Selezione** (dopo il blocco precedente): scegli, tra `nodo.hijos`, il figlio con il miglior `ucb_score`, e prosegui la simulazione ricorsivamente su di esso, salvando il risultato nella variabile `v`

**Suggerimenti**

- Per l'espansione: scorri `movimientos_validos(nodo.tablero)`, applica ogni mossa con `aplicar_movimiento(nodo.tablero, m, nodo.jugador)` e crea il figlio con `Nodo(tablero_hijo, -nodo.jugador)` — nota che il figlio eredita il turno del giocatore avversario
- Salva ogni figlio così: `nodo.hijos[m] = Nodo(tablero_hijo, -nodo.jugador)`
- Per la selezione: `mejor_m = max(nodo.hijos, key=lambda m: ucb_score(nodo, nodo.hijos[m]))` ti dà la chiave del figlio migliore
- Prosegui la simulazione con una chiamata ricorsiva: `v = simular(nodo.hijos[mejor_m])`

**Cosa aspettarti mentre completi l'esercizio**: con i due `# TODO` vuoti, `busqueda_mcts(tablero, jugador, num_simulaciones=1)` non fallisce (fa solo un'espansione della radice), ma restituisce un dizionario di visite vuoto. Appena passi a più di una simulazione, se l'espansione è ancora vuota vedrai ancora un dizionario vuoto; se hai completato l'espansione ma non la selezione, vedrai un errore Python che segnala una variabile mancante — è il segnale che devi completare la selezione.

In [ ]:
def simular(nodo):
    """Un ciclo de seleccion, expansion, evaluacion y retropropagacion."""
    r = resultado(nodo.tablero)
    if r is not None:
        nodo.N += 1
        nodo.W += r
        return r

    if not nodo.expandido:
        # TODO: expansion. Para cada movimiento valido en nodo.tablero, crea
        # un Nodo hijo con el tablero resultante y el jugador contrario, y
        # guardalo en nodo.hijos[m]. No olvides poner nodo.expandido = True
        # al final.

        # Evaluacion con la tabla de valores/heuristica (esto ya esta hecho)
        v = valor_estimado(nodo.tablero)
        nodo.N += 1
        nodo.W += v
        return v

    # TODO: seleccion. Elige, de entre nodo.hijos, el hijo con mejor
    # ucb_score(nodo, hijo), y continua la simulacion recursivamente sobre
    # el, guardando el resultado en la variable `v`.

    # Retropropagacion (esto ya esta hecho, una vez tengas `v`)
    nodo.N += 1
    nodo.W += v
    return v


def busqueda_mcts(tablero, jugador, num_simulaciones=60):
    """Devuelve {movimiento: visitas} tras num_simulaciones simulaciones."""
    raiz = Nodo(tablero, jugador)
    for _ in range(num_simulaciones):
        simular(raiz)
    return {m: hijo.N for m, hijo in raiz.hijos.items()}


**Esegui la cella seguente per provare la tua implementazione.** Con la scacchiera vuota e 60 simulazioni, la casella 4 (il centro) dovrebbe accumulare chiaramente più visite delle altre, perché partecipa a quattro linee vincenti.

In [ ]:
visitas_prueba = busqueda_mcts(tablero_nuevo(), 1, num_simulaciones=60)
print("Visitas por movimiento:", visitas_prueba)


<details>
<summary>💡 Vedi la soluzione</summary>

```python
def simular(nodo):
    """Un ciclo di selezione, espansione, valutazione e retropropagazione."""
    r = resultado(nodo.tablero)
    if r is not None:
        nodo.N += 1
        nodo.W += r
        return r

    if not nodo.expandido:
        # Espansione: un figlio per ogni mossa valida
        for m in movimientos_validos(nodo.tablero):
            tablero_hijo = aplicar_movimiento(nodo.tablero, m, nodo.jugador)
            nodo.hijos[m] = Nodo(tablero_hijo, -nodo.jugador)
        nodo.expandido = True

        v = valor_estimado(nodo.tablero)
        nodo.N += 1
        nodo.W += v
        return v

    # Selezione: scendiamo verso il figlio con il punteggio UCT migliore
    mejor_m = max(nodo.hijos, key=lambda m: ucb_score(nodo, nodo.hijos[m]))
    v = simular(nodo.hijos[mejor_m])

    nodo.N += 1
    nodo.W += v
    return v
```
</details>

## Esercizio 2 — Adatta lo scheletro a Forza 4 ridotto

Ora riutilizzi lo stesso schema —autogioco, MCTS, tabella dei valori— in un gioco diverso: una versione ridotta di Forza 4, su una scacchiera 4x4, in cui vinci collegando tre pedine in linea (invece di quattro, come nel vero Forza 4).

**Perché tu possa andare avanti anche se l'esercizio uno non è ancora completato**, la cella seguente contiene una copia già completa e funzionante del motore di ricerca (`Nodo`, `ucb_score`, `simular`, `busqueda_mcts`, `elegir_movimiento`, autogioco e valutazione), e contiene anche già adattati la scacchiera, le mosse valide (con gravità: una pedina cade nella casella libera più bassa della sua colonna, come nel vero Forza 4) e l'euristica.

**L'unica cosa che ti tocca completare è `resultado(tablero)`**, la funzione che decide se qualcuno ha vinto. Qui sotto trovi la firma con un `# TODO` e il controllo del pareggio già risolto.

**Suggerimenti**

- La logica è la stessa del Tris visto in classe: scorri `LINEAS_GANADORAS` (già generate per te, per la scacchiera 4x4) e somma i tre valori di ogni linea
- Collegare tre pedine proprie somma +3 (vince X) o -3 (vince O), esattamente come nel Tris, perché una linea è sempre composta da tre caselle
- Se nessuna linea dà +3 né -3, continua a controllare: la scacchiera potrebbe essere piena (pareggio, già risolto) oppure la partita potrebbe continuare (`None`, già risolto)

**Cosa aspettarti mentre completi l'esercizio**: con il `# TODO` vuoto, `resultado` non rileva mai una vittoria, quindi tutte le partite della cella di confronto finiranno in pareggio dopo aver riempito la scacchiera (16 mosse). Non dà errori, ma non ti permette di vedere l'effetto reale della ricerca: è il segnale che ti manca ancora da completare il controllo della vittoria.

In [ ]:
# --- Conecta 4 reducido: tablero 4x4, conecta tres en linea ---
FILAS, COLUMNAS = 4, 4


def indice(fila, col):
    return fila * COLUMNAS + col


def generar_lineas_ganadoras_4x4():
    """Todas las lineas de tres casillas seguidas: horizontales, verticales
    y las dos diagonales, dentro de un tablero de 4x4."""
    lineas = []
    for f in range(FILAS):
        for c in range(COLUMNAS - 2):
            lineas.append((indice(f, c), indice(f, c + 1), indice(f, c + 2)))
    for c in range(COLUMNAS):
        for f in range(FILAS - 2):
            lineas.append((indice(f, c), indice(f + 1, c), indice(f + 2, c)))
    for f in range(FILAS - 2):
        for c in range(COLUMNAS - 2):
            lineas.append((indice(f, c), indice(f + 1, c + 1), indice(f + 2, c + 2)))
    for f in range(FILAS - 2):
        for c in range(2, COLUMNAS):
            lineas.append((indice(f, c), indice(f + 1, c - 1), indice(f + 2, c - 2)))
    return lineas


LINEAS_GANADORAS = generar_lineas_ganadoras_4x4()


def tablero_nuevo():
    return tuple([0] * (FILAS * COLUMNAS))


def movimientos_validos(tablero):
    # una columna es valida si su casilla superior todavia esta vacia
    return [c for c in range(COLUMNAS) if tablero[indice(0, c)] == 0]


def aplicar_movimiento(tablero, col, jugador):
    """La ficha cae a la casilla libre mas baja de la columna elegida."""
    nuevo = list(tablero)
    for f in range(FILAS - 1, -1, -1):
        if nuevo[indice(f, col)] == 0:
            nuevo[indice(f, col)] = jugador
            break
    return tuple(nuevo)


def resultado(tablero):
    """Devuelve +1 si gana X, -1 si gana O, 0 si hay empate, None si sigue."""
    # TODO: recorre LINEAS_GANADORAS y comprueba si alguna linea suma +3
    # (gana X) o -3 (gana O). Pista: suma = tablero[a] + tablero[b] + tablero[c]

    if 0 not in tablero:
        return 0
    return None


def heuristica(tablero):
    puntuacion = 0
    for linea in LINEAS_GANADORAS:
        valores = [tablero[i] for i in linea]
        piezas_x = valores.count(1)
        piezas_o = valores.count(-1)
        if piezas_o == 0:
            puntuacion += piezas_x ** 2
        if piezas_x == 0:
            puntuacion -= piezas_o ** 2
    return math.tanh(puntuacion / 6.0)


tabla_valores = {}


def valor_estimado(tablero):
    if tablero in tabla_valores:
        return tabla_valores[tablero]
    return heuristica(tablero)


In [ ]:
# --- Motor de busqueda: copia completa y funcionando, para que puedas
# avanzar aunque el ejercicio uno no este terminado ---
class Nodo:
    __slots__ = ("tablero", "jugador", "hijos", "N", "W", "expandido")

    def __init__(self, tablero, jugador):
        self.tablero = tablero
        self.jugador = jugador
        self.hijos = {}
        self.N = 0
        self.W = 0.0
        self.expandido = False

    def valor_medio(self):
        return self.W / self.N if self.N > 0 else 0.0


def ucb_score(padre, hijo, c=1.4):
    q = hijo.valor_medio() * padre.jugador
    u = c * math.sqrt(math.log(padre.N + 1) / (hijo.N + 1))
    return q + u


def simular(nodo):
    r = resultado(nodo.tablero)
    if r is not None:
        nodo.N += 1
        nodo.W += r
        return r

    if not nodo.expandido:
        for m in movimientos_validos(nodo.tablero):
            tablero_hijo = aplicar_movimiento(nodo.tablero, m, nodo.jugador)
            nodo.hijos[m] = Nodo(tablero_hijo, -nodo.jugador)
        nodo.expandido = True
        v = valor_estimado(nodo.tablero)
        nodo.N += 1
        nodo.W += v
        return v

    mejor_m = max(nodo.hijos, key=lambda m: ucb_score(nodo, nodo.hijos[m]))
    v = simular(nodo.hijos[mejor_m])
    nodo.N += 1
    nodo.W += v
    return v


def busqueda_mcts(tablero, jugador, num_simulaciones):
    raiz = Nodo(tablero, jugador)
    for _ in range(num_simulaciones):
        simular(raiz)
    return {m: hijo.N for m, hijo in raiz.hijos.items()}


def elegir_movimiento(visitas, temperatura):
    movimientos = list(visitas.keys())
    conteos = np.array([visitas[m] for m in movimientos], dtype=float)
    if temperatura == 0:
        return movimientos[int(np.argmax(conteos))]
    probabilidades = conteos ** (1.0 / temperatura)
    probabilidades /= probabilidades.sum()
    return np.random.choice(movimientos, p=probabilidades)


def jugar_partida_autojuego(num_simulaciones, temperatura=0.8):
    tablero = tablero_nuevo()
    jugador = 1
    historial = []
    while resultado(tablero) is None:
        visitas = busqueda_mcts(tablero, jugador, num_simulaciones)
        movimiento = elegir_movimiento(visitas, temperatura)
        historial.append(tablero)
        tablero = aplicar_movimiento(tablero, movimiento, jugador)
        jugador = -jugador
    return historial, resultado(tablero)


def actualizar_tabla_valores(historial, resultado_final, alpha=0.5):
    for tablero in historial:
        actual = valor_estimado(tablero)
        tabla_valores[tablero] = actual + alpha * (resultado_final - actual)


def movimiento_aleatorio(tablero):
    return random.choice(movimientos_validos(tablero))


def jugar_vs_aleatorio(num_simulaciones):
    tablero = tablero_nuevo()
    jugador = 1
    while resultado(tablero) is None:
        if jugador == 1:
            visitas = busqueda_mcts(tablero, jugador, num_simulaciones)
            movimiento = elegir_movimiento(visitas, temperatura=0)
        else:
            movimiento = movimiento_aleatorio(tablero)
        tablero = aplicar_movimiento(tablero, movimiento, jugador)
        jugador = -jugador
    return resultado(tablero)


**Esegui la cella seguente per confrontare tre valori di `num_simulaciones` (10, 30 e 60).** Per ciascuno, addestra sei generazioni (40 partite di autogioco, 50 partite di valutazione per generazione, come in classe) e osserva in quale generazione le sconfitte arrivano a zero. Con `resultado` non completato, vedrai sconfitte e vittorie a zero per tutto l'addestramento —solo pareggi— in tutte e tre le configurazioni: questo è l'indizio che il TODO è ancora in sospeso.

In [ ]:
for num_sim in [10, 30, 60]:
    tabla_valores.clear()
    np.random.seed(0)
    random.seed(0)

    print(f"--- num_simulaciones = {num_sim} ---")
    for generacion in range(1, 7):
        for _ in range(40):
            historial, resultado_final = jugar_partida_autojuego(num_sim)
            actualizar_tabla_valores(historial, resultado_final)

        victorias, empates, derrotas = 0, 0, 0
        for _ in range(50):
            r = jugar_vs_aleatorio(num_sim)
            if r == 1:
                victorias += 1
            elif r == 0:
                empates += 1
            else:
                derrotas += 1

        print(f"  Generacion {generacion}: {victorias} victorias, {empates} empates, {derrotas} derrotas")
    print()


<details>
<summary>💡 Vedi la soluzione</summary>

```python
def resultado(tablero):
    """Restituisce +1 se vince X, -1 se vince O, 0 se c'è pareggio, None se la partita continua."""
    for a, b, c in LINEAS_GANADORAS:
        suma = tablero[a] + tablero[b] + tablero[c]
        if suma == 3:
            return 1
        if suma == -3:
            return -1

    if 0 not in tablero:
        return 0
    return None
```

Con questa soluzione, in un'esecuzione di riferimento (seed impostato a 0) lo schema è chiaro: con `num_simulaciones=10` compaiono ancora alcune sconfitte fino a fine addestramento (per esempio, 5 sconfitte nella generazione 1 e altre 5 nella generazione 5, e le sconfitte non arrivano a zero fino alla generazione 6); con `num_simulaciones=30` le sconfitte arrivano a zero nella maggior parte delle generazioni, con qualche eccezione puntuale; e con `num_simulaciones=60` l'agente smette di perdere contro il giocatore casuale già dalla generazione 1. Meno simulazioni MCTS fa l'agente a ogni mossa, peggiore è la politica che quella ricerca produce, e più autogioco serve per compensarla, esattamente come nel secondo esperimento suggerito nel manuale del corso.
</details>

## Per continuare a esercitarti

**Prova a portare `FILAS` e `COLUMNAS` a 5x5, mantenendo la connessione di tre in linea, e riesegui il confronto su `num_simulaciones`.** L'agente ha bisogno di più simulazioni o di più generazioni per smettere di perdere contro il giocatore casuale? Discutine con il tuo team: è la stessa domanda che si sono posti i team di DeepMind passando dal Tris giocattolo al vero Go, solo che qui l'intero esperimento gira in pochi secondi.

Con questo si conclude il corso completo di Reinforcement Learning (RL) da zero. **Complimenti, e continua a esplorare.**